## 4. Prep Watersheds (Arctic Rivers)

Joins outlet segment stats to watershed polygons and simplifies for web delivery.

**Outlet logic:** One outlet per watershed — the segment with the highest ma99 (mean annual flow)
per watershed, as determined by `join_segments_gages_watersheds.ipynb`. Stored in
`segments/arctic_rivers_segments_joined_3338.shp` as rows where `outlet == 1`.
140 unique outlet COMIDs cover 142 watersheds (2 watersheds share one outlet segment).

**Input:**
- `watersheds/arctic_rivers_watersheds_3338.shp`
- `segments/arctic_rivers_segments_joined_3338.shp` (outlet flags)
- `arctic_stats.csv` (outlet stats)

**Output:** `export/arctic_rivers_watersheds_stats_simplified.shp`

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [ ]:
WATERSHEDS_PATH = "watersheds/arctic_rivers_watersheds_3338.shp"
JOINED_PATH     = "segments/arctic_rivers_segments_joined_3338.shp"
STATS_CSV       = "arctic_stats.csv"
OUTPUT_PATH     = "export/arctic_rivers_watersheds_stats_simplified.shp"
SIMPLIFY_TOL    = 500  # metres (EPSG:3338 is metric)

In [3]:
watersheds = gpd.read_file(WATERSHEDS_PATH)
joined     = gpd.read_file(JOINED_PATH)
stats      = pd.read_csv(STATS_CSV, index_col="stream_id")

print(f"Watersheds: {len(watersheds)} rows, cols: {list(watersheds.columns)}")
print(f"Joined:     {len(joined)} rows, cols: {list(joined.columns)}")
print(f"Stats:      {len(stats)} rows, {len(stats.columns)} columns")

Watersheds: 142 rows, cols: ['ID_1', 'ID_2', 'Name', 'geometry']
Joined:     34497 rows, cols: ['COMID', 'Gauge_ID', 'ID_1', 'ID_2', 'Name', 'outlet', 'geometry']
Stats:      34346 rows, 66 columns


In [4]:
# One row per watershed: take the outlet segment (outlet==1), deduplicate on ID_1
# to handle the 2 watersheds that share an outlet COMID.
outlets = (
    joined[joined["outlet"] == 1]
    .drop_duplicates(subset="ID_1")[["ID_1", "COMID"]]
    .reset_index(drop=True)
)
print(f"Outlet rows: {len(outlets)}  |  unique COMIDs: {outlets['COMID'].nunique()}")

Outlet rows: 142  |  unique COMIDs: 127


In [5]:
# Attach stats for each outlet COMID
outlets_with_stats = outlets.merge(stats, left_on="COMID", right_index=True, how="left")

missing = outlets_with_stats[stats.columns].isna().all(axis=1).sum()
if missing:
    print(f"Warning: {missing} outlet(s) have no stats")

# Merge stats onto watershed polygons via ID_1
merged = watersheds.merge(outlets_with_stats.drop(columns="geometry", errors="ignore"),
                          on="ID_1", how="left")
print(f"Merged watershed polygons: {len(merged)}")

Merged watershed polygons: 142


In [6]:
# Simplify watershed polygons
merged["geometry"] = merged.simplify(SIMPLIFY_TOL, preserve_topology=True)

# Make an export directory if one doesn't exist
!mkdir -p export

merged.to_file(OUTPUT_PATH)
print(f"Wrote {OUTPUT_PATH}  ({len(merged)} watersheds)")

Wrote segments/arctic_rivers_watersheds_stats_simplified.shp  (142 watersheds)
